# Task 3: Transformer-Based Modeling

This notebook implements the transformer-based modeling approach using a pretrained BERT model.

The three strategies implemented are:

1. **Strategy 1 — Domain Only Fine-Tuning**
2. **Strategy 2 — Sequential Transfer Learning**
3. **Strategy 3 — Mixed Training**

The models are compared quantitatively using Accuracy, Precision, Recall, and F1-score.


## Setup 1: Libraries

This training process requires a number of libraries for the management of the data and training of the transformer models. The libraries are needed are for the BERT pre-trained model, PyTorch training processes, data handling and processing, and final model evaluation.


In [ ]:
#Install necessary libraries if not already done so
!pip install transformers torch scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
#Import libraries and modules for the notebook
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader


## Setup 2: Loading and Processing the Three Datasets

This modelling process will use the three pre-prepared datasets from task 1. Though each dataset has been generated prior to this notebook, it is important that they are checked and validated at least somewhat to ensure that changes in the data pipeline do not significantly affect this notebook, where code can take a long time to run and errors can be difficult to fix.

**Dataset Summary:**

As specified in the project description, models must be trained on three datasets, and evaluated on the domain specific dataset. The datasets chosen, as generated in task 1, are as follows:

- `goemotions_5class.csv` = General dataset, from the Google GoEmotions dataset.
- `fpb_5class.csv` = Domain specific dataset, from the Financial Phrasebank dataset (Financial domain).
- `combined_5class.csv` = Mixed dataset, combining the previous two datasets.

The assignment requires comparing general/domain/mixed training strategies, and the datasets are loaded hence. The general and mixed datasets are not split into testing and validation sets as all models are evaluated against the domain specific dataset.

### Loading

Each dataset is packaged as a separate .csv file. Though data preparation has been completed already in task 1, it is appropriate to check and adjust the provided dataset to ensure it works with this notebook.


In [ ]:
#Read dataset files into pandas dataframes
general_df = pd.read_csv("../data/processed/goemotions_5class.csv")
domain_train_df = pd.read_csv("../data/processed/fpb_train.csv")
# domain_val_df = pd.read_csv("../data/processed/fpb_val.csv")
domain_test_df = pd.read_csv("../data/processed/fpb_test.csv")
combined_df = pd.read_csv("../data/processed/combined_5class.csv")

#Check dataframe shapes
print("General dataset shape:", general_df.shape)
print("Domain train dataset shape:", domain_train_df.shape)
# print("Domain validation dataset shape:", domain_val_df.shape)
print("Domain test dataset shape:", domain_test_df.shape)
print("Combined dataset shape:", combined_df.shape)

#Display header values from each dataframe
display(general_df.head())
display(domain_train_df.head())
# display(domain_val_df.head())
display(domain_test_df.head())
display(combined_df.head())


General dataset shape: (43404, 4)
Domain train dataset shape: (3392, 4)
Domain validation dataset shape: (727, 4)
Domain test dataset shape: (727, 4)
Combined dataset shape: (48250, 3)


,text,label,source,clean_text
0,my favourite food is anything i didn't have to...,Neutral,GoEmotions,my favourite food is anything i didn t have to...
1,"now if he does off himself, everyone will thin...",Neutral,GoEmotions,now if he does off himself everyone will think...
2,why the fuck is bayless isoing,Fear,GoEmotions,why the fuck is bayless isoing
3,to make her feel threatened,Fear,GoEmotions,to make her feel threatened
4,dirty southern wankers,Fear,GoEmotions,dirty southern wankers


,text,label,source,clean_text
0,"in stead of being based on a soft drink , as i...",Neutral,FinancialPhraseBank,in stead of being based on a soft drink as is ...
1,"look out for vintage fabric cushion covers , '...",Neutral,FinancialPhraseBank,look out for vintage fabric cushion covers NUM...
2,"thanks to my nokia and lulu , i am now proud t...",Neutral,FinancialPhraseBank,thanks to my nokia and lulu i am now proud to ...
3,nordstjernan has used its option to buy anothe...,Neutral,FinancialPhraseBank,nordstjernan has used its option to buy anothe...
4,25 november 2010 - finnish paints and coatings...,Neutral,FinancialPhraseBank,NUM november NUM finnish paints and coatings c...


,text,label,source,clean_text
0,the shares represented 4.998 of total share ca...,Neutral,FinancialPhraseBank,the shares represented NUM NUM of total share ...
1,the center will be built in the kapuli distric...,Neutral,FinancialPhraseBank,the center will be built in the kapuli distric...
2,"svyturys-utenos alus , which is controlled by ...",Optimism,FinancialPhraseBank,svyturys utenos alus which is controlled by th...
3,"under the agreement , tietoenator will provide...",Optimism,FinancialPhraseBank,under the agreement tietoenator will provide b...
4,we will continue to make it easier for users t...,Neutral,FinancialPhraseBank,we will continue to make it easier for users t...


,text,label,source,clean_text
0,the share capital of alma media corporation bu...,Neutral,FinancialPhraseBank,the share capital of alma media corporation bu...
1,the eu commission said earlier it had fined th...,Fear,FinancialPhraseBank,the eu commission said earlier it had fined th...
2,"kesko pursues a strategy of healthy , focused ...",Optimism,FinancialPhraseBank,kesko pursues a strategy of healthy focused gr...
3,down to eur5 .9 m h1 '09 3 august 2009 - finni...,Sadness,FinancialPhraseBank,down to eurNUM NUM m hNUM NUM NUM august NUM f...
4,"cencorp would focus on the development , manuf...",Neutral,FinancialPhraseBank,cencorp would focus on the development manufac...


,text,label,source
0,my favourite food is anything i didn't have to...,Neutral,GoEmotions
1,"now if he does off himself, everyone will thin...",Neutral,GoEmotions
2,why the fuck is bayless isoing,Fear,GoEmotions
3,to make her feel threatened,Fear,GoEmotions
4,dirty southern wankers,Fear,GoEmotions


### Checking Column Names

Before training, we must confirm the text and label columns in each dataset is correct to ensure the data has been loaded properly. Mismatches here will prevent data from being processed and models from being trained later.

In [ ]:
#Print column names from each dataframe
print("General columns:", general_df.columns.tolist())
print("Domain training columns:", domain_train_df.columns.tolist())
# print("Domain validation columns:", domain_val_df.columns.tolist())
print("Domain testing columns:", domain_test_df.columns.tolist())
print("Combined columns:", combined_df.columns.tolist())

General columns: ['text', 'label', 'source', 'clean_text']
Domain training columns: ['text', 'label', 'source', 'clean_text']
Domain validation columns: ['text', 'label', 'source', 'clean_text']
Domain testing columns: ['text', 'label', 'source', 'clean_text']
Combined columns: ['text', 'label', 'source']


In [ ]:
#Set column names to check for based on those generated previously
text_column = "text"
label_column = "label"

### Cleaning Dataset Rows

Incorrectly loaded dataset rows must be removed before the models can be trained. Additionally, it is imperative that each data entry is loaded with a consistent data type, specifically a string.

In [ ]:
#Drop rows with missing values
general_df = general_df[[text_column, label_column]].dropna()
domain_train_df = domain_train_df[[text_column, label_column]].dropna()
# domain_val_df = domain_val_df[[text_column, label_column]].dropna()
domain_test_df = domain_test_df[[text_column, label_column]].dropna()
combined_df = combined_df[[text_column, label_column]].dropna()

#Save "text_column" data as string type
general_df[text_column] = general_df[text_column].astype(str)
domain_train_df[text_column] = domain_train_df[text_column].astype(str)
# domain_val_df[text_column] = domain_val_df[text_column].astype(str)
domain_test_df[text_column] = domain_test_df[text_column].astype(str)
combined_df[text_column] = combined_df[text_column].astype(str)

#Print dataframe shapes. Each should only have two columns
print("General dataset after cleaning:", general_df.shape)
print("Domain training dataset after cleaning:", domain_train_df.shape)
# print("Domain validation dataset after cleaning:", domain_val_df.shape)
print("Domain testing dataset after cleaning:", domain_test_df.shape)
print("Combined dataset after cleaning:", combined_df.shape)


General dataset after cleaning: (43404, 2)
Domain training dataset after cleaning: (3392, 2)
Domain validation dataset after cleaning: (727, 2)
Domain testing dataset after cleaning: (727, 2)
Combined dataset after cleaning: (48250, 2)


### Checking Label Values
 
For model training purposes, labels must be numeric. We need to check how labels are stored in the dataframes and convert them to numeric data if they are not already.

In [ ]:
#Print unique values from each datasets labels column. If these are strings, they must be converted to numbers
print("General labels:", general_df[label_column].unique())
print("Domain labels:", domain_train_df[label_column].unique())
# print("Domain labels:", domain_val_df[label_column].unique())
print("Domain labels:", domain_test_df[label_column].unique())
print("Combined labels:", combined_df[label_column].unique())


General labels: <ArrowStringArray>
['Neutral', 'Fear', 'Joy', 'Optimism', 'Sadness']
Length: 5, dtype: str
Domain labels: <ArrowStringArray>
['Neutral', 'Optimism', 'Sadness', 'Joy', 'Fear']
Length: 5, dtype: str
Domain labels: <ArrowStringArray>
['Neutral', 'Optimism', 'Sadness', 'Fear', 'Joy']
Length: 5, dtype: str
Domain labels: <ArrowStringArray>
['Neutral', 'Fear', 'Optimism', 'Sadness', 'Joy']
Length: 5, dtype: str
Combined labels: <ArrowStringArray>
['Neutral', 'Fear', 'Joy', 'Optimism', 'Sadness']
Length: 5, dtype: str


### Converting Labels to Numbers

BERT needs numeric class labels for training. By mapping each label to a numeric value (an integer in this case), we can train the model on the dataset where it would not have been possible before.

Numerical assignments can be generated progressively, but because we only have a few labels, five to be exact, we can define a dictionary that maps each label to a number. A more sophisticated deployment may manage these labels programmatically instead of hard-coding them, but this notebook is not intended for general applicability.


In [ ]:
#Define dictionary for mapping labels to numbers
label_map = {
    "neutral": 0,
    "optimism": 1,
    "sadness": 2,
    "joy": 3,
    "fear": 4
}

#Function to convert values in the given dataframe to their numeric equivalents
def convert_labels(df, label_column):
    # If labels are already numeric, keep them as integers
    if pd.api.types.is_numeric_dtype(df[label_column]):
        df[label_column] = df[label_column].astype(int)
        return df

    # Otherwise map text labels to numbers. Ensure strings have no leading spaces or capitals
    df[label_column] = df[label_column].astype(str).str.lower().str.strip()
    df[label_column] = df[label_column].map(label_map)
    
    #Drop missing values if not converted properly, and save labels as integers
    df = df.dropna()
    df[label_column] = df[label_column].astype(int)
    return df

#Convert dataframes to their numerically labelled equivalents
general_df = convert_labels(general_df, label_column)
domain_train_df = convert_labels(domain_train_df, label_column)
# domain_val_df = convert_labels(domain_val_df, label_column)
domain_test_df = convert_labels(domain_test_df, label_column)
combined_df = convert_labels(combined_df, label_column)

print("General final shape:", general_df.shape)
print("Domain final shape:", domain_train_df.shape)
# print("Domain final shape:", domain_val_df.shape)
print("Domain final shape:", domain_test_df.shape)
print("Combined final shape:", combined_df.shape)
print("\n")
print("General label counts:")
print(general_df[label_column].value_counts())
print("\n")
print("Domain training set label counts:")
print(domain_train_df[label_column].value_counts())
# print("\n")
# print("Domain validation set label counts:")
# print(domain_val_df[label_column].value_counts())
print("\n")
print("Domain testing set label counts:")
print(domain_test_df[label_column].value_counts())
print("\n")
print("Combined label counts:")
print(combined_df[label_column].value_counts())


General final shape: (43404, 2)
Domain final shape: (3392, 2)
Domain final shape: (727, 2)
Domain final shape: (727, 2)
Combined final shape: (48250, 2)


General label counts:
label
0    17312
1     9302
3     7624
4     6520
2     2646
Name: count, dtype: int64


Domain training set label counts:
label
0    2015
1     912
2     391
3      42
4      32
Name: count, dtype: int64


Domain validation set label counts:
label
0    432
1    195
2     84
3      9
4      7
Name: count, dtype: int64


Domain testing set label counts:
label
0    432
1    196
2     83
3      9
4      7
Name: count, dtype: int64


Combined label counts:
label
0    20191
1    10605
3     7684
4     6566
2     3204
Name: count, dtype: int64


### Loading the Tokeniser

BERT cannot read raw text directly. To manage this, the tokeniser converts text into the BERT input format. The BERT base uncased tokeniser is used as case variation is not important for the semantic interpretation of the data here.

In [ ]:
#Load the tokeniser, using "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

### Creating the Dataset Class

BERT requires data to be input in a particulary way for the model to be trained. Notably, the text sample must have:
- input IDs
- an attention mask
- a label

This class prepares each text sample as such, allowing BERT to train on the data.


In [ ]:
#Define SentimentDataset class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts[index])
        label = int(self.labels[index])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "label": torch.tensor(label, dtype=torch.long)
        }


With the data loaded, checked, and tokenised, the modelling part of the task can be done safely.

## Setup 3: Modelling Setup

A few bespoke functions need to be created to train the transformer models with the project's specific requirements. These include the model training function, and the evaluation function.

### Training Function

This function will perform one full training pass with the provided datasets.

In [ ]:
#Define the model training function
def train_model(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    return total_loss / len(dataloader)


### Evaluation Function

The assignment requires quantitative comparison, so statistics describing model performance must be generated. This function calculates the models accuracy, precision, recall, and F1-score. These will be compared and collected later on for comparison with the other modelling approaches.


In [ ]:
#Define the model evaluation function
def evaluate_model(model, dataloader, device):
    model.eval()

    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = torch.argmax(outputs.logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("\nClassification Report:\n")
    print(classification_report(true_labels, predictions, zero_division=0))
    # print("True labels unique:", set(true_labels))
    # print("Predictions unique:", set(predictions))
    # print("Num true labels:", len(true_labels))
    # print("Num predictions:", len(predictions))
    # print(np.unique(true_labels, return_counts=True))
    # print(np.unique(predictions, return_counts=True))
    print("\nConfusion Matrix:\n")
    print(confusion_matrix(true_labels, predictions))

    return accuracy, precision, recall, f1


### Setting Device and Hyperparameters

Transformer models train faster on GPUs than they do on CPUs. Depending on the system this program is run on, certain resources may or may not be available to the trainer. In this case, the following device options are:
- `cpu`: Available to all systems
- `cuda`: Available to systems with CUDA GPUs
- `mps`: Available to systems running on Apple Silicon

Aside from the device chosen, certain hyperparameters must be chosen for the training process. These may be adjusted to gain better model performance through a hyperparameter tuning process.

In [50]:
# Set MPS as default device
import torch

# Check device availability and pick the best available. Run on CPU if no better option exists
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Set default tensor type for MPS
if device.type == "mps":
    torch.set_default_tensor_type('torch.FloatTensor')

#Define hyperparameters
MAX_LEN = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
EPOCHS = 3
NUM_LABELS = 5

print("Using device:", device)


Using device: mps


With data processing and modelling processes established, the training and evaluation can now be done.

## Strategy 1: Domain Only Fine-Tuning

The BERT model will first be trained on the domain dataset. This will demonstrate the models performance for domain-on-domain modelling.

In [ ]:
####____________________________________

# domain_train_texts, domain_val_texts, domain_train_labels, domain_val_labels = train_test_split(
#     domain_df[text_column].to_numpy(),
#     domain_df[label_column].to_numpy(),
#     test_size=0.2,
#     random_state=42,
#     stratify=domain_df[label_column].to_numpy()
# )

# domain_train_dataset = SentimentDataset(domain_train_texts, domain_train_labels, tokenizer, MAX_LEN)
# domain_val_dataset = SentimentDataset(domain_val_texts, domain_val_labels, tokenizer, MAX_LEN)

domain_train_dataset = SentimentDataset(domain_train_df["text"], domain_train_df["label"], tokenizer, MAX_LEN)
domain_test_dataset = SentimentDataset(domain_test_df["text"], domain_test_df["label"], tokenizer, MAX_LEN)

domain_train_loader = DataLoader(domain_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
domain_test_loader = DataLoader(domain_test_dataset, batch_size=BATCH_SIZE)

print("Domain training samples:", len(domain_train_dataset))
print("Domain validation samples:", len(domain_test_dataset))


Domain training samples: 3392
Domain validation samples: 727


In [ ]:
model_domain = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_domain.to(device)

optimizer = AdamW(model_domain.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_domain, domain_train_loader, optimizer, device)
    print("Training Loss:", train_loss)
    domain_only_results = evaluate_model(model_domain, domain_test_loader, device)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9102.94it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Epoch 1/3
Training Loss: 0.8178040441344766
Accuracy: 0.7997054491899853
Precision: 0.812049126818517
Recall: 0.7997054491899853
F1 Score: 0.796656205697196

Classification Report:

              precision    recall  f1-score   support

           0       0.93      0.79      0.85       403
           1       0.63      0.88      0.74       183
           2       0.76      0.83      0.80        78
           3       0.00      0.00      0.00         9
           4       0.00      0.00      0.00         6

    accuracy                           0.80       679
   macro avg       0.47      0.50      0.48       679
weighted avg       0.81      0.80      0.80       679


Confusion Matrix:

[[317  74  12   0   0]
 [ 17 161   5   0   0]
 [  4   9  65   0   0]
 [  0   9   0   0   0]
 [  2   1   3   0   0]]
Epoch 2/3
Training Loss: 0.39974838582908406
Accuracy: 0.8541973490427098
Precision: 0.8438365062076403
Recall: 0.8541973490427098
F1 Score: 0.8468405943523891

Classification Report:

        

## Strategy 2: Sequential Transfer Learning

Train on the general dataset first, then fine-tune on the domain dataset.

**Why we use this:**  
This is the main transfer learning experiment. The model first learns general sentiment patterns, then adapts to domain-specific language.


In [61]:
# general_train_texts, general_val_texts, general_train_labels, general_val_labels = train_test_split(
#     general_df[text_column].to_numpy(),
#     general_df[label_column].to_numpy(),
#     test_size=0.2,
#     random_state=42,
#     stratify=general_df[label_column].to_numpy()
# )

# general_train_dataset = SentimentDataset(general_train_texts, general_train_labels, tokenizer, MAX_LEN)
# general_train_loader = DataLoader(general_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

general_train_dataset = SentimentDataset(general_df["text"], general_df["label"], tokenizer, MAX_LEN)
general_train_loader = DataLoader(general_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("General training samples:", len(general_train_dataset))


General training samples: 43404


In [ ]:
model_sequential = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_sequential.to(device)

optimizer = AdamW(model_sequential.parameters(), lr=LEARNING_RATE)

print("Stage 1: Training on general dataset")

for epoch in range(EPOCHS):
    print(f"General Training Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_sequential, general_train_loader, optimizer, device)
    print("General Training Loss:", train_loss)
    
#Run time: 56m approx on fin's macbook

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14565.84it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

Stage 1: Training on general dataset
General Training Epoch 1/3
General Training Loss: 0.868816264251694
General Training Epoch 2/3
General Training Loss: 0.6600617980322823
General Training Epoch 3/3
General Training Loss: 0.46750355934709625


In [ ]:
optimizer = AdamW(model_sequential.parameters(), lr=LEARNING_RATE)

print("Stage 2: Fine-tuning on domain dataset")

for epoch in range(EPOCHS):
    print(f"Domain Fine-Tuning Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_sequential, domain_train_loader, optimizer, device)
    print("Domain Fine-Tuning Loss:", train_loss)
    sequential_results = evaluate_model(model_sequential, domain_test_loader, device)

#Run time: xx approx on fin's macbook

Stage 2: Fine-tuning on domain dataset
Domain Fine-Tuning Epoch 1/3
Domain Fine-Tuning Loss: 0.5032146983615164
Accuracy: 0.8360824742268042
Precision: 0.8368902710977123
Recall: 0.8360824742268042
F1 Score: 0.8342963462444629

Classification Report:

              precision    recall  f1-score   support

           0       0.90      0.87      0.88       576
           1       0.72      0.83      0.77       261
           2       0.87      0.79      0.83       112
           3       0.43      0.25      0.32        12
           4       0.33      0.11      0.17         9

    accuracy                           0.84       970
   macro avg       0.65      0.57      0.59       970
weighted avg       0.84      0.84      0.83       970

True labels unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Predictions unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Num true labels: 970
Num predictions: 970
(array([0, 1, 2, 3, 4]), array([576, 261, 112

## Strategy 3: Mixed Training

Train BERT once using the combined dataset.

**Why we use this:**  
This tests whether combining general and domain data gives more stable performance.


In [ ]:
# combined_train_texts, combined_val_texts, combined_train_labels, combined_val_labels = train_test_split(
#     combined_df[text_column].values,
#     combined_df[label_column].values,
#     test_size=0.2,
#     random_state=42,
#     stratify=combined_df[label_column].values
# )

# combined_train_dataset = SentimentDataset(combined_train_texts, combined_train_labels, tokenizer, MAX_LEN)
# combined_val_dataset = SentimentDataset(combined_val_texts, combined_val_labels, tokenizer, MAX_LEN)

# combined_train_loader = DataLoader(combined_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# combined_val_loader = DataLoader(combined_val_dataset, batch_size=BATCH_SIZE)

combined_train_dataset = SentimentDataset(combined_df["text"], combined_df["label"], tokenizer, MAX_LEN)
# combined_val_dataset = SentimentDataset(combined_val_texts, combined_val_labels, tokenizer, MAX_LEN)

combined_train_loader = DataLoader(combined_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# combined_val_loader = DataLoader(combined_val_dataset, batch_size=BATCH_SIZE)

print("Combined training samples:", len(combined_train_dataset))
# print("Combined validation samples:", len(combined_val_dataset))


Combined training samples: 38600
Combined validation samples: 9650


In [ ]:
model_mixed = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=NUM_LABELS
)

model_mixed.to(device)

optimizer = AdamW(model_mixed.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    print(f"Mixed Training Epoch {epoch + 1}/{EPOCHS}")
    train_loss = train_model(model_mixed, combined_train_loader, optimizer, device)
    print("Mixed Training Loss:", train_loss)
    mixed_results = evaluate_model(model_mixed, domain_test_loader, device)

#Run time: xx approx on fin's macbook

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8990.67it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Mixed Training Epoch 1/3
Mixed Training Loss: 0.832143371709392
Accuracy: 0.7179274611398964
Precision: 0.7165703085437325
Recall: 0.7179274611398964
F1 Score: 0.7167202944490103

Classification Report:

              precision    recall  f1-score   support

           0       0.76      0.76      0.76      4038
           1       0.67      0.69      0.68      2121
           2       0.67      0.58      0.62       641
           3       0.75      0.81      0.78      1537
           4       0.64      0.59      0.62      1313

    accuracy                           0.72      9650
   macro avg       0.70      0.69      0.69      9650
weighted avg       0.72      0.72      0.72      9650

True labels unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Predictions unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Num true labels: 9650
Num predictions: 9650
(array([0, 1, 2, 3, 4]), array([4038, 2121,  641, 1537, 1313]))
(array([0, 1, 2, 3, 4]), ar

# Final Quantitative Comparison

All models are compared on the **same domain validation dataset**.

**Why we use this:**  
The project requires comparing the Transformer strategies quantitatively. Using the same domain validation set makes the comparison fair.


In [ ]:
print("Final Evaluation: Domain Only Model")
domain_only_results = evaluate_model(model_domain, domain_test_loader, device)

print("\nFinal Evaluation: Sequential Transfer Learning Model")
sequential_results = evaluate_model(model_sequential, domain_test_loader, device)

print("\nFinal Evaluation: Mixed Training Model")
mixed_results_on_domain = evaluate_model(model_mixed, domain_test_loader, device)


Final Evaluation: Domain Only Model
Accuracy: 0.8546391752577319
Precision: 0.8478985601103771
Recall: 0.8546391752577319
F1 Score: 0.8489087870535842

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.94      0.90       576
           1       0.85      0.75      0.80       261
           2       0.82      0.79      0.81       112
           3       0.25      0.17      0.20        12
           4       0.33      0.11      0.17         9

    accuracy                           0.85       970
   macro avg       0.63      0.55      0.58       970
weighted avg       0.85      0.85      0.85       970

True labels unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Predictions unique: {np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)}
Num true labels: 970
Num predictions: 970
(array([0, 1, 2, 3, 4]), array([576, 261, 112,  12,   9]))
(array([0, 1, 2, 3, 4]), array([622, 229, 108,   8,   3]))

Co

In [ ]:
results_df = pd.DataFrame({
    "Strategy": [
        "Strategy 1 - Domain Only Fine-Tuning",
        "Strategy 2 - Sequential Transfer Learning",
        "Strategy 3 - Mixed Training"
    ],
    "Accuracy": [
        domain_only_results[0],
        sequential_results[0],
        mixed_results_on_domain[0]
    ],
    "Precision": [
        domain_only_results[1],
        sequential_results[1],
        mixed_results_on_domain[1]
    ],
    "Recall": [
        domain_only_results[2],
        sequential_results[2],
        mixed_results_on_domain[2]
    ],
    "F1 Score": [
        domain_only_results[3],
        sequential_results[3],
        mixed_results_on_domain[3]
    ]
})

results_df


,Strategy,Accuracy,Precision,Recall,F1 Score
0,Strategy 1 - Domain Only Fine-Tuning,0.854639,0.847899,0.854639,0.848909
1,Strategy 2 - Sequential Transfer Learning,0.834021,0.831807,0.834021,0.832359
2,Strategy 3 - Mixed Training,0.957732,0.957959,0.957732,0.957655


## Save the Best Model

**Why we use this:**  
The best model can be reused later without training again.


In [ ]:
best_model = model_sequential

best_model.save_pretrained("best_bert_task3_model")
tokenizer.save_pretrained("best_bert_task3_model")

print("Best model saved successfully.")


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]

Best model saved successfully.


## Short Report Explanation

For Task 3, a pretrained BERT model was used for Transformer-based sentiment classification. Three training strategies were implemented and compared. The first strategy fine-tuned BERT only on the domain dataset. The second strategy used sequential transfer learning by first training BERT on a general sentiment dataset and then fine-tuning it on the domain dataset. The third strategy trained BERT once on a combined dataset containing both general and domain samples. Each strategy was evaluated using accuracy, precision, recall, and F1-score to determine how training data choice influenced domain-specific sentiment classification performance.
